# WASTE CLASSIFICATION — COMPLETE COLAB MANUAL
## 05 — Ifaza M.D.U — IT25102486
### Pixel Normalization

**IMPORTANT:** This version is intentionally code-heavy. Each numbered cell below is intended to be copied into Google Colab and run in sequence. Do not skip validation cells.

## A. ONE-TIME COLAB SETUP
### 00. Set member identity — run this before the common cells

In [ ]:
MEMBER_NAME = "Ifaza M.D.U"
IT_NUMBER = "IT25102486"
TASK = "Pixel Normalization"
print(MEMBER_NAME, IT_NUMBER, TASK)

### 01. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
print("Google Drive mounted successfully.")

### 02. Install/check packages

In [ ]:
!pip -q install pandas pillow matplotlib numpy imagehash
import os, shutil, hashlib, json, csv, math
from pathlib import Path
import pandas as pd
import numpy as np
from PIL import Image, ImageFile
import matplotlib.pyplot as plt

ImageFile.LOAD_TRUNCATED_IMAGES = False
print("Packages imported.")

### 03. Define project folders

In [ ]:
PROJECT = Path("/content/drive/MyDrive/Waste_Classification")
RAW_DIR = PROJECT / "data" / "raw"
PROCESSED_DIR = PROJECT / "data" / "processed"
RESULTS_DIR = PROJECT / "results"
REPORT_DIR = PROJECT / "audit_reports"
EDA_DIR = RESULTS_DIR / "eda_visualizations"
OUTPUT_DIR = RESULTS_DIR / "outputs"

for p in [PROCESSED_DIR, RESULTS_DIR, REPORT_DIR, EDA_DIR, OUTPUT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

IMAGE_EXTS = {".jpg",".jpeg",".png",".webp",".bmp",".gif",".tif",".tiff"}
print("PROJECT =", PROJECT)
print("RAW_DIR =", RAW_DIR)
assert PROJECT.exists(), "Waste_Classification folder was not found."
assert RAW_DIR.exists(), "data/raw was not found. Check your Drive folder."

### 04. List project folders

In [ ]:
for p in [PROJECT, RAW_DIR, PROCESSED_DIR, RESULTS_DIR, REPORT_DIR]:
    print("\n", p)
    if p.exists():
        for child in list(p.iterdir())[:20]:
            print("  ", child.name)

### 05. Create a run log

In [ ]:
RUN_LOG = OUTPUT_DIR / "preprocessing_run_log.txt"
RUN_LOG.write_text(
    "Waste Classification preprocessing run\n"
    f"Member: {MEMBER_NAME}\n"
    f"IT Number: {IT_NUMBER}\n"
)
print("Run log:", RUN_LOG)

## B. YOUR COMPLETE ASSIGNED-PREPROCESSING CODE
### 06. Set input stage

In [ ]:
STAGE4 = PROCESSED_DIR / "stage4_standardized"
assert STAGE4.exists(), "Stage 4 is missing."
files = sorted(STAGE4.rglob("*.jpg"))
print("Input images:", len(files))

### 07. Install/import PyTorch

In [ ]:
import torch
from torchvision import transforms
print("PyTorch:", torch.__version__)

### 08. Define ImageNet normalization

In [ ]:
IMAGE_SIZE = (224,224)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

to_tensor = transforms.ToTensor()
normalize = transforms.Normalize(
    mean=IMAGENET_MEAN,
    std=IMAGENET_STD
)
print("Mean:", IMAGENET_MEAN)
print("Std :", IMAGENET_STD)

### 09. Test one image before/after normalization

In [ ]:
img=Image.open(files[0]).convert("RGB")
tensor_before=to_tensor(img)
tensor_after=normalize(tensor_before.clone())

print("Image:",files[0].name)
print("Before min/max/mean:",float(tensor_before.min()),float(tensor_before.max()),float(tensor_before.mean()))
print("After min/max/mean:",float(tensor_after.min()),float(tensor_after.max()),float(tensor_after.mean()))
print("Tensor shape:",tensor_after.shape)

### 10. Plot normalization EDA

In [ ]:
plt.figure(figsize=(9,5))
plt.hist(tensor_before.flatten().numpy(),bins=50,alpha=.6,label="Before")
plt.hist(tensor_after.flatten().numpy(),bins=50,alpha=.6,label="After")
plt.title("Pixel Values Before vs After ImageNet Normalization")
plt.xlabel("Tensor value")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()
path=EDA_DIR/"member5_normalization_histogram.png"
plt.savefig(path,dpi=200); plt.show()
print("Saved:",path)

### 11. Verify transform on a batch

In [ ]:
sample_files=files[:min(16,len(files))]
batch=torch.stack([normalize(to_tensor(Image.open(p).convert("RGB"))) for p in sample_files])
print("Batch shape:",batch.shape)
print("Batch mean:",float(batch.mean()))
print("Batch std:",float(batch.std()))
assert batch.shape[1:] == (3,224,224)

### 12. Save exact preprocessing specification

In [ ]:
spec = {
    "resize": [224,224],
    "to_tensor": True,
    "normalize_mean": IMAGENET_MEAN,
    "normalize_std": IMAGENET_STD,
    "note": "Normalization is a model-input transform; do not save normalized values as JPEG."
}
path=OUTPUT_DIR/"member5_normalization_spec.json"
path.write_text(json.dumps(spec,indent=2))
print("Saved:",path)

### 13. Create reusable transform code file

In [ ]:
transform_code = """from torchvision import transforms

train_or_eval_normalize = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])
"""
path=OUTPUT_DIR/"member5_normalization_transform.py"
path.write_text(transform_code)
print("Saved:",path)

### 14. Git save commands

In [ ]:
!git -C /content/drive/MyDrive/Waste_Classification status
!git -C /content/drive/MyDrive/Waste_Classification add notebooks results/outputs results/eda_visualizations
!git -C /content/drive/MyDrive/Waste_Classification commit -m "IT25102486 - normalization transform"
# !git -C /content/drive/MyDrive/Waste_Classification push

## C. FINAL CHECKS — RUN THESE AFTER YOUR LAST STAGE CELL
### FINAL CHECK 1 — Count the output images

In [ ]:
# Change this only if your stage output folder has a different name.
STAGE_OUT = {
    "01": PROCESSED_DIR / "stage1_cleaning",
    "02": PROCESSED_DIR / "stage2_deduplicated",
    "03": PROCESSED_DIR / "stage3_labels",
    "04": PROCESSED_DIR / "stage4_standardized",
    "05": PROCESSED_DIR / "stage4_standardized",
    "06": PROCESSED_DIR / "stage4_standardized"
}["05"]

if STAGE_OUT.exists():
    output_images=[p for p in STAGE_OUT.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTS]
    print("Output images:",len(output_images))
else:
    print("This stage does not create a new permanent image folder.")

### FINAL CHECK 2 — List your output files

In [ ]:
print("Outputs:")
for p in sorted(OUTPUT_DIR.glob("*")):
    print(" -", p.name)

print("\nEDA:")
for p in sorted(EDA_DIR.glob("*")):
    print(" -", p.name)

### FINAL CHECK 3 — Save a completion record

In [ ]:
completion = {
    "member": MEMBER_NAME,
    "it_number": IT_NUMBER,
    "task": TASK,
    "status": "COMPLETED",
    "next_step": "Hand off verified output to the next member."
}
completion_path = OUTPUT_DIR / "IT25102486_completion_record.json"
completion_path.write_text(json.dumps(completion, indent=2))
print("Saved:", completion_path)

### FINAL CHECK 4 — Git status

In [ ]:
!git -C /content/drive/MyDrive/Waste_Classification status

### FINAL CHECK 5 — What you hand to the next member

In [ ]:
print("""
=======================================================
MEMBER 05 — PIXEL NORMALIZATION HANDOFF SUMMARY
=======================================================
Input Stage  : data/processed/stage4_standardized
Output Stage : data/processed/stage4_standardized (No new image folder saved; transform used at input)
Image Size   : 224 x 224 (RGB)
Normalization: ImageNet Benchmark
  Mean       : [0.485, 0.456, 0.406]
  Std        : [0.229, 0.224, 0.225]
Tensor Shape : (3, 224, 224)

Artifacts Generated:
- Notebook  : notebooks/IT25102486_PixelNormalization.ipynb
- Transform : results/outputs/member5_normalization_transform.py
- Spec JSON : results/outputs/member5_normalization_spec.json
- Histogram : results/eda_visualizations/member5_normalization_histogram.png
- Record    : results/outputs/IT25102486_completion_record.json
=======================================================
""")